# Programa para juntar a base SCOPUS e WOS para análise de Abstract e Título

In [ ]:
import pandas as pd
import re

## 1. Arquivos de entrada

In [ ]:
arquivo_scopus = "base_scopus_consolidada.xlsx"
arquivo_wos = "base_wos_consolidada.txt"

## 2. Função para parsear o formato WOS (formato tagged)

In [ ]:
def parse_wos_file(filepath):
    """
    Parseia o arquivo WOS no formato tagged do Clarivate Analytics.
    Cada registro termina com 'ER' (End of Record).
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Divide por 'ER' como marcador de fim de registro
    records = re.split(r'\nER\s*\n', content)
    
    dados = []
    
    for record in records:
        if not record.strip():
            continue
        
        linhas = record.split('\n')
        
        # Encontra o índice onde começam os dados (após FN e VR)
        start_idx = 0
        for i, linha in enumerate(linhas):
            if linha.startswith('TI '):
                start_idx = i
                break
        
        titulo = ''
        autores = []
        abstract = ''
        
        i = start_idx
        while i < len(linhas):
            linha = linhas[i]
            
            # Title - pode ter continuação nas próximas linhas
            if linha.startswith('TI '):
                titulo = linha[3:].strip()
                j = i + 1
                while j < len(linhas):
                    prox = linhas[j]
                    if prox.startswith(' '):
                        titulo += ' ' + prox.strip()
                        j += 1
                    else:
                        break
            
            # Authors - múltiplas linhas AU
            elif linha.startswith('AU '):
                autores.append(linha[3:].strip())
            
            # Abstract - pode ter continuação
            elif linha.startswith('AB '):
                abstract = linha[3:].strip()
                j = i + 1
                while j < len(linhas):
                    prox = linhas[j]
                    # Para se encontrar outra tag principal
                    if prox and len(prox) >= 2 and prox[:2] in ['PT', 'SO', 'LA', 'DT', 'ID', 'TI', 'AU', 'C1', 'RP', 'CR', 'FX']:
                        break
                    # Se começa com espaço, é continuação do abstract
                    if prox.startswith(' '):
                        abstract += ' ' + prox.strip()
                        j += 1
                    else:
                        break
            
            i += 1
        
        if titulo:
            dados.append({
                'titulo': titulo,
                'autores': '; '.join(autores),
                'abstract': abstract
            })
    
    return pd.DataFrame(dados)

## 3. Leitura das bases

In [ ]:
# Leitura da base SCOPUS
df_scopus = pd.read_excel(arquivo_scopus)
print(f"SCOPUS: {len(df_scopus)} registros")
print(f"Colunas SCOPUS: {df_scopus.columns.tolist()[:10]}")

In [ ]:
# Leitura e parsing da base WOS
df_wos = parse_wos_file(arquivo_wos)
print(f"WOS: {len(df_wos)} registros")

## 4. Mapeamento das colunas SCOPUS

In [ ]:
# Função para localizar colunas equivalentes
def encontrar_coluna(df, candidatos):
    cols_lower = {c.lower().strip(): c for c in df.columns}
    for cand in candidatos:
        cand_norm = cand.lower().strip()
        if cand_norm in cols_lower:
            return cols_lower[cand_norm]
    return None

# Possíveis nomes de colunas em cada base
candidatos_titulo = ["titulo", "título", "title", "ti", "article title", "document title"]
candidatos_autores = ["autores", "authors", "au", "author(s)", "authors"]
candidatos_abstract = ["abstract", "resumo", "ab", "summary"]

# Mapeamento SCOPUS
col_titulo_s = encontrar_coluna(df_scopus, candidatos_titulo)
col_autores_s = encontrar_coluna(df_scopus, candidatos_autores)
col_abstract_s = encontrar_coluna(df_scopus, candidatos_abstract)

print(f"Coluna título SCOPUS: {col_titulo_s}")
print(f"Coluna autores SCOPUS: {col_autores_s}")
print(f"Coluna abstract SCOPUS: {col_abstract_s}")

## 5. Padronização das colunas principais

In [ ]:
# Padronização SCOPUS
scopus_padrao = pd.DataFrame({
    "titulo": df_scopus[col_titulo_s] if col_titulo_s else "",
    "autores": df_scopus[col_autores_s] if col_autores_s else "",
    "abstract": df_scopus[col_abstract_s] if col_abstract_s else "",
})

# WOS já está padronizado pelo parser
wos_padrao = pd.DataFrame({
    "titulo": df_wos['titulo'] if 'titulo' in df_wos.columns else "",
    "autores": df_wos['autores'] if 'autores' in df_wos.columns else "",
    "abstract": df_wos['abstract'] if 'abstract' in df_wos.columns else "",
})

print(f"SCOPUS padronizado: {len(scopus_padrao)} registros")
print(f"WOS padronizado: {len(wos_padrao)} registros")

## 6. Junção das bases

In [ ]:
# Junta as bases
df_final = pd.concat([scopus_padrao, wos_padrao], ignore_index=True)
print(f"Total após junção: {len(df_final)} registros")

## 7. Adiciona colunas para triagem (PRISMA)

In [ ]:
# Adiciona colunas para preenchimento posterior
colunas_triagem = [
    "STATUS_TRIAGEM",
    "IDENTIFICADOR_DUPLICADO",
    "CRITERIO_EXCLUSAO",
    "RELEVANCIA",
    "DECISAO_FINAL",
    "OBS",
]

for col in colunas_triagem:
    df_final[col] = ""

print(f"Colunas adicionadas: {colunas_triagem}")

## 8. Visualização e salvamento

In [ ]:
# Visualiza as primeiras linhas
print("\nPrimeiros registros:")
print(df_final.head())

print("\nColunas do DataFrame final:")
print(df_final.columns.tolist())

In [ ]:
# Salva em xlsx
arquivo_saida = "base_consolidada_para_triagem.xlsx"

df_final.to_excel(arquivo_saida, index=False)

print(f"\nArquivo salvo com sucesso: {arquivo_saida}")
print(f"Total de registros: {len(df_final)}")